In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [5]:
df = pd.read_csv('/content/sample_data/hiring.csv')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   experience                  6 non-null      object 
 1   test_score(out of 10)       7 non-null      float64
 2   interview_score(out of 10)  8 non-null      int64  
 3   salary($)                   8 non-null      int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 388.0+ bytes


In [7]:

# Fill missing experience with 'zero'
df['experience'] = df['experience'].fillna('zero')

# Convert text experience to numeric
exp_map = {
    'zero':0, 'one':1, 'two':2, 'three':3,
    'four':4, 'five':5, 'six':6, 'seven':7,
    'eight':8, 'nine':9, 'ten':10,
    'eleven':11, 'twelve':12
}
df['experience'] = df['experience'].map(exp_map)

# Fill missing test scores with median
df['test_score(out of 10)'].fillna(
    df['test_score(out of 10)'].median(),
    inplace=True
)

# -------------------- Model Building --------------------

X = df[['experience',
        'test_score(out of 10)',
        'interview_score(out of 10)']]

y = df['salary($)']

model = LinearRegression()
model.fit(X, y)

# Print regression equation
print("Regression Equation (Built-in):")
print(f"Salary = {model.coef_[0]:.2f}*Experience + "
      f"{model.coef_[1]:.2f}*TestScore + "
      f"{model.coef_[2]:.2f}*InterviewScore + "
      f"{model.intercept_:.2f}")

# -------------------- Predictions --------------------

salary1 = model.predict([[2,9,6]])
salary2 = model.predict([[12,10,10]])

print("\nPredicted Salary (2 yr, 9 test, 6 interview):", salary1[0])
print("Predicted Salary (12 yr, 10 test, 10 interview):", salary2[0])

Regression Equation (Built-in):
Salary = 2812.95*Experience + 1845.71*TestScore + 2205.24*InterviewScore + 17737.26

Predicted Salary (2 yr, 9 test, 6 interview): 53205.96797671033
Predicted Salary (12 yr, 10 test, 10 interview): 92002.18340611353


/tmp/ipython-input-107394249.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['test_score(out of 10)'].fillna(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [8]:
df['experience'] = df['experience'].fillna('zero')

exp_map = {
    'zero':0, 'one':1, 'two':2, 'three':3,
    'four':4, 'five':5, 'six':6, 'seven':7,
    'eight':8, 'nine':9, 'ten':10,
    'eleven':11, 'twelve':12
}
df['experience'] = df['experience'].map(exp_map)

df['test_score(out of 10)'].fillna(
    df['test_score(out of 10)'].median(),
    inplace=True
)

X = df[['experience',
        'test_score(out of 10)',
        'interview_score(out of 10)']].values

y = df['salary($)'].values

X = np.c_[np.ones(X.shape[0]), X]

beta = np.linalg.inv(X.T.dot(X)).dot(X.T).dot(y)

# Print regression equation
print("Regression Equation (Without sklearn):")
print(f"Salary = {beta[1]:.2f}*Experience + "
      f"{beta[2]:.2f}*TestScore + "
      f"{beta[3]:.2f}*InterviewScore + "
      f"{beta[0]:.2f}")

# Candidate 1
candidate1 = np.array([1, 2, 9, 6])
salary1 = candidate1.dot(beta)

# Candidate 2
candidate2 = np.array([1, 12, 10, 10])
salary2 = candidate2.dot(beta)

print("\nPredicted Salary (2 yr, 9 test, 6 interview):", salary1)
print("Predicted Salary (12 yr, 10 test, 10 interview):", salary2)

Regression Equation (Without sklearn):
Salary = nan*Experience + nan*TestScore + nan*InterviewScore + nan

Predicted Salary (2 yr, 9 test, 6 interview): nan
Predicted Salary (12 yr, 10 test, 10 interview): nan


/tmp/ipython-input-2184944213.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['test_score(out of 10)'].fillna(


In [9]:
df2 = pd.read_csv('/content/sample_data/1000_Companies.csv')

In [10]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        1000 non-null   float64
 1   Administration   1000 non-null   float64
 2   Marketing Spend  1000 non-null   float64
 3   State            1000 non-null   object 
 4   Profit           1000 non-null   float64
dtypes: float64(4), object(1)
memory usage: 39.2+ KB


In [21]:
# # One Hot Encoding for 'State' (drop first to avoid dummy variable trap)
# df2 = pd.get_dummies(df2, columns=['State'], drop_first=True)

# Define features (X) and target (y)
X = df2.drop('Profit', axis=1)
y = df2['Profit']

# -------------------- Model Building --------------------

model = LinearRegression()
model.fit(X, y)

# Print regression equation
print("Regression Equation (Built-in):")
equation = "Profit = "
for col, coef in zip(X.columns, model.coef_):
    equation += f"{coef:.2f}*{col} + "
equation += f"{model.intercept_:.2f}"
print(equation)

# -------------------- Prediction --------------------

# Example: Predict Profit for
# R&D Spend = 91694.48, Administration = 515841.3, Marketing Spend = 11931.24, State = Florida

# Determine dummy values for State columns
# Assume columns after one-hot encoding are: State_Florida, State_New York
# Since Florida → State_Florida = 1, State_New York = 0
new_data = [[91694.48, 515841.3, 11931.24, 1, 0]]  # column order matches X.columns

predicted_profit = model.predict(new_data)
print("\nPredicted Profit:", predicted_profit[0])

Regression Equation (Built-in):
Profit = 0.55*R&D Spend + 1.03*Administration + 0.08*Marketing Spend + -446.35*State_Florida + 97.73*State_New York + -70051.25

Predicted Profit: 510570.9926108309


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
